# Notebook 02: Building Tools

Creating tools with OpenAI function calling format.

## 1. Setup

In [10]:
import os
import json
import random
from typing import Dict, List
from dotenv import load_dotenv
from openai import OpenAI
import boto3

load_dotenv()

LLM_BASE_URL = os.getenv('LLM_BASE_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o')
KNOWLEDGE_BASE_ID = os.getenv('KNOWLEDGE_BASE_ID')

llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

print('[OK] Configuration loaded')

[OK] Configuration loaded


## 2. Tool Schema Format

In [11]:
# OpenAI function calling schema structure
example_schema = {
    "type": "function",
    "function": {
        "name": "tool_name",
        "description": "What this tool does - helps LLM decide when to use it",
        "parameters": {
            "type": "object",
            "properties": {
                "param1": {"type": "string", "description": "Parameter description"}
            },
            "required": ["param1"]
        }
    }
}
print(json.dumps(example_schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "tool_name",
    "description": "What this tool does - helps LLM decide when to use it",
    "parameters": {
      "type": "object",
      "properties": {
        "param1": {
          "type": "string",
          "description": "Parameter description"
        }
      },
      "required": [
        "param1"
      ]
    }
  }
}


## 3. Tool 1: Search Knowledge Base (Uses retrieve() API)

This tool wraps the same `bedrock_agent.retrieve()` from Notebook 01.
Now it's ONE tool among many - the agent decides when to use it vs other tools.

In [12]:
SEARCH_KB_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Search the company knowledge base for policies, FAQs, shipping, and returns.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    }
}

def search_knowledge_base(query: str) -> Dict:
    """Search Bedrock Knowledge Base"""
    try:
        response = bedrock_agent.retrieve(
            knowledgeBaseId=KNOWLEDGE_BASE_ID,
            retrievalQuery={'text': query},
            retrievalConfiguration={'vectorSearchConfiguration': {'numberOfResults': 3}}
        )
        results = [{'content': r['content']['text'], 'source': r['location']['s3Location']['uri'].split('/')[-1]} 
                   for r in response.get('retrievalResults', [])]
        return {'success': True, 'results': results, 'count': len(results)}
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Test
result = search_knowledge_base("return policy")
print(f"Success: {result['success']}, Results: {result['count']}")

Success: True, Results: 3


## 4. Tool 2: Check Order Status (Mock - Bypasses retrieve())

Unlike `retrieve()` which searches documents, this queries an order database directly.
Returns actual order data instead of policy text.

In [13]:
MOCK_ORDERS = {
    "ORD-12345": {"status": "shipped", "carrier": "FedEx", "tracking": "794644790138", 
                  "estimated_delivery": "2026-02-03", "items": ["Blue iPhone 15 Case"], "destination": "New York, NY"},
    "ORD-67890": {"status": "processing", "carrier": None, "estimated_delivery": "2026-02-05", 
                  "items": ["Wireless Earbuds"], "destination": "Miami, FL"},
    "ORD-11111": {"status": "delivered", "carrier": "UPS", "delivered_date": "2026-01-28", 
                  "items": ["Laptop Stand"], "destination": "Los Angeles, CA"}
}

CHECK_ORDER_SCHEMA = {
    "type": "function",
    "function": {
        "name": "check_order_status",
        "description": "Check customer order status, tracking, or delivery ETA.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID (e.g., ORD-12345)"}
            },
            "required": ["order_id"]
        }
    }
}

def check_order_status(order_id: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"):
        order_id = f"ORD-{order_id}"
    if order_id in MOCK_ORDERS:
        return {'success': True, 'order_id': order_id, **MOCK_ORDERS[order_id]}
    return {'success': False, 'error': f'Order {order_id} not found'}

# Test
print(json.dumps(check_order_status("ORD-12345"), indent=2))

{
  "success": true,
  "order_id": "ORD-12345",
  "status": "shipped",
  "carrier": "FedEx",
  "tracking": "794644790138",
  "estimated_delivery": "2026-02-03",
  "items": [
    "Blue iPhone 15 Case"
  ],
  "destination": "New York, NY"
}


## 5. Tool 3: Get Weather Alerts (Mock - External API)

Unlike `retrieve()` which has static indexed documents, this calls a weather API.
Returns current conditions, not what was true when docs were indexed.

In [14]:
MOCK_WEATHER = {
    "miami": {"location": "Miami, FL", "condition": "Hurricane Warning", "alert_level": "severe", 
              "shipping_impact": "2-3 day delay", "estimated_delay_days": 3},
    "new york": {"location": "New York, NY", "condition": "Clear", "alert_level": "none", 
                 "shipping_impact": "No delays", "estimated_delay_days": 0},
    "chicago": {"location": "Chicago, IL", "condition": "Winter Storm", "alert_level": "moderate", 
                "shipping_impact": "1 day delay", "estimated_delay_days": 1}
}

GET_WEATHER_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_weather_alerts",
        "description": "Check weather and shipping alerts for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and state (e.g., Miami, FL)"}
            },
            "required": ["location"]
        }
    }
}

def get_weather_alerts(location: str) -> Dict:
    key = location.lower().split(',')[0].strip()
    if key in MOCK_WEATHER:
        return {'success': True, **MOCK_WEATHER[key]}
    return {'success': True, 'location': location, 'condition': 'Clear', 'estimated_delay_days': 0}

# Test
print(json.dumps(get_weather_alerts("Miami, FL"), indent=2))

{
  "success": true,
  "location": "Miami, FL",
  "condition": "Hurricane Warning",
  "alert_level": "severe",
  "shipping_impact": "2-3 day delay",
  "estimated_delay_days": 3
}


## 6. Tool 4: Check Inventory (Mock - Live Database)

Unlike `retrieve()` which may have outdated catalog info, this checks live stock levels.
Inventory changes every second - documents can't keep up.

In [15]:
MOCK_INVENTORY = {
    "iphone 15 case": {
        "blue": {"in_stock": True, "quantity": 42, "price": 29.99},
        "black": {"in_stock": False, "quantity": 0, "restock_date": "2026-02-10"}
    },
    "airpods pro": {"default": {"in_stock": True, "quantity": 120, "price": 199.00}},
    "wireless earbuds": {
        "white": {"in_stock": True, "quantity": 85, "price": 49.99},
        "black": {"in_stock": True, "quantity": 63, "price": 49.99}
    }
}

CHECK_INVENTORY_SCHEMA = {
    "type": "function",
    "function": {
        "name": "check_inventory",
        "description": "Check product availability and stock levels.",
        "parameters": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string", "description": "Product name"},
                "color": {"type": "string", "description": "Color variant (optional)"}
            },
            "required": ["product_name"]
        }
    }
}

def check_inventory(product_name: str, color: str = None) -> Dict:
    key = product_name.lower().strip()
    for pkey in MOCK_INVENTORY:
        if pkey in key or key in pkey:
            data = MOCK_INVENTORY[pkey]
            if color and color.lower() in data:
                return {'success': True, 'product': pkey.title(), 'color': color, **data[color.lower()]}
            return {'success': True, 'product': pkey.title(), 'variants': data}
    return {'success': False, 'error': 'Product not found'}

# Test
print(json.dumps(check_inventory("iPhone 15 case", "blue"), indent=2))

{
  "success": true,
  "product": "Iphone 15 Case",
  "color": "blue",
  "in_stock": true,
  "quantity": 42,
  "price": 29.99
}


## 7. Tool 5: Create Return Request (Mock - Takes Action)

Unlike `retrieve()` which is read-only, this CREATES a record.
This is the key difference: retrieve() finds information, this tool takes action.

In [16]:
CREATE_RETURN_SCHEMA = {
    "type": "function",
    "function": {
        "name": "create_return_request",
        "description": "Create a return request for an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID to return"},
                "reason": {"type": "string", "description": "Reason for return"}
            },
            "required": ["order_id", "reason"]
        }
    }
}

def create_return_request(order_id: str, reason: str) -> Dict:
    order_id = order_id.upper().strip()
    if not order_id.startswith("ORD-"):
        order_id = f"ORD-{order_id}"
    if order_id not in MOCK_ORDERS:
        return {'success': False, 'error': 'Order not found'}
    if MOCK_ORDERS[order_id]['status'] == 'processing':
        return {'success': False, 'error': 'Order not shipped yet'}
    return_id = f"RET-{random.randint(10000, 99999)}"
    return {
        'success': True, 'return_id': return_id, 'order_id': order_id, 'reason': reason,
        'label_url': f'https://returns.example.com/{return_id}',
        'instructions': 'Print label, pack items, drop at FedEx/UPS'
    }

# Test
print(json.dumps(create_return_request("ORD-12345", "defective"), indent=2))

{
  "success": true,
  "return_id": "RET-34561",
  "order_id": "ORD-12345",
  "reason": "defective",
  "label_url": "https://returns.example.com/RET-34561",
  "instructions": "Print label, pack items, drop at FedEx/UPS"
}


## 8. Tool Registry

In [17]:
TOOL_REGISTRY = {
    "search_knowledge_base": {"schema": SEARCH_KB_SCHEMA, "function": search_knowledge_base},
    "check_order_status": {"schema": CHECK_ORDER_SCHEMA, "function": check_order_status},
    "get_weather_alerts": {"schema": GET_WEATHER_SCHEMA, "function": get_weather_alerts},
    "check_inventory": {"schema": CHECK_INVENTORY_SCHEMA, "function": check_inventory},
    "create_return_request": {"schema": CREATE_RETURN_SCHEMA, "function": create_return_request}
}

def get_all_tool_schemas() -> List[Dict]:
    return [tool["schema"] for tool in TOOL_REGISTRY.values()]

def execute_tool(tool_name: str, arguments: Dict) -> Dict:
    if tool_name not in TOOL_REGISTRY:
        return {"error": f"Unknown tool: {tool_name}"}
    return TOOL_REGISTRY[tool_name]["function"](**arguments)

print("Tools:", list(TOOL_REGISTRY.keys()))

Tools: ['search_knowledge_base', 'check_order_status', 'get_weather_alerts', 'check_inventory', 'create_return_request']


## 9. Test All Tools

In [18]:
print("Testing all tools:")
print("1. KB Search:", execute_tool("search_knowledge_base", {"query": "shipping"}).get('count', 'N/A'), "results")
print("2. Order Status:", execute_tool("check_order_status", {"order_id": "ORD-12345"}).get('status', 'N/A'))
print("3. Weather:", execute_tool("get_weather_alerts", {"location": "Miami"}).get('condition', 'N/A'))
print("4. Inventory:", execute_tool("check_inventory", {"product_name": "AirPods"}).get('product', 'N/A'))
print("5. Return:", execute_tool("create_return_request", {"order_id": "ORD-12345", "reason": "defective"}).get('return_id', 'N/A'))

Testing all tools:
1. KB Search: 3 results
2. Order Status: shipped
3. Weather: Hurricane Warning
4. Inventory: Airpods Pro
5. Return: RET-97734


## Summary

Built 5 tools:
- `search_knowledge_base` - Wraps `retrieve()` API (same as baseline RAG)
- `check_order_status` - Queries order database (bypasses retrieve)
- `get_weather_alerts` - Calls weather API (bypasses retrieve)
- `check_inventory` - Checks live stock (bypasses retrieve)
- `create_return_request` - Creates records (action, not just retrieval)

The agent will choose which tool to use based on the user's question.

**Next:** Build the ReAct agent that uses these tools.